# Pillar H — stochastic / robust optimization under grade uncertainty

This notebook reproduces **H4 — Stochastic / robust optimization under grade uncertainty**. H4's exec DoD:
on synthetic blocks with a known truth grade and a posterior stub whose `.samples` returns noisy draws
around it, the *stochastic* (CVaR-averse) extraction plan's expected value on held-out truth scenarios is
`>=` the deterministic-mean plan's, and its CVaR (downside) is strictly better.

**Judgment call.** `mixle.stochastic_opt` (H4's own module: `StochasticPlan`, `two_stage_stochastic_plan`,
`cvar_epigraph`) has not landed yet. We fall back to a reference implementation of the exact DR-ALG
(Rockafellar-Uryasev CVaR epigraph as a MILP), but it is built on the **real, already-landed**
`mixle.relations.branch_and_bound_milp` -- the same mixed-integer solver H4's own algorithm calls for --
so the optimization core here is not reinvented, only the thin `stochastic_opt` wiring is.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from collections import namedtuple

from mixle.relations import branch_and_bound_milp  # real, already-landed MILP solver H4 builds on

try:
    from mixle.stochastic_opt import StochasticPlan, two_stage_stochastic_plan, cvar_epigraph
    HAVE_REAL_H4 = True
except ImportError:
    HAVE_REAL_H4 = False

print("using landed mixle.stochastic_opt:", HAVE_REAL_H4)

## 1. Load the committed fixture

12 blocks; `g` is 30 training scenario draws of grade per block (the IC-1 `.samples(k_scenarios, rng)` the real H4 would draw from a fitted posterior), `g_test` is 400 independent held-out truth scenarios used only for out-of-sample evaluation, never for fitting the plan.

In [ ]:
data = np.load("../../../data/pillar_validation/h_stochastic_blocks.npz")
true_grade, cost, sigma = data["true_grade"], data["cost"], data["sigma"]
price = float(data["price"])
g, g_test = data["g"], data["g_test"]
n_blocks, K = g.shape
alpha, lam = 0.9, 1.0
print(f"{n_blocks} blocks, {K} training scenarios, {g_test.shape[0]} held-out truth scenarios")

## 2. The CVaR epigraph (Rockafellar-Uryasev) + the two-stage stochastic plan

`cvar_epigraph` emits the `eta`/`u_k` variable block and constraint rows for `CVaR_alpha(L) = min_eta [eta + (1/((1-alpha)K)) sum_k u_k]`, `u_k >= L_k(x) - eta`, `u_k >= 0`. `two_stage_stochastic_plan` assembles `maximize E_k[v_k(x)] - lambda * CVaR_alpha(-v_k(x))` as one MILP and solves it with `branch_and_bound_milp`.

In [ ]:
StochasticPlanFallback = namedtuple("StochasticPlan", ["extract", "expected_value", "cvar", "scenarios"])


def fallback_cvar_epigraph(losses, alpha):
    """H4 `cvar_epigraph(losses, alpha) -> (c_add, a_ub_rows, b_ub, var_index)`.

    `losses[k, b]` is scenario k's loss coefficient on decision x_b (`L_k(x) = losses[k] @ x`); returns the
    extra `eta` (free) + `u_1..u_K` (>=0) objective coefficients and the epigraph constraint rows to append
    to the base block-selection MILP. `var_index` is eta's column within the appended block.
    """
    K, n = losses.shape
    n_extra = 1 + K
    c_add = np.zeros(n_extra)
    c_add[0] = 1.0
    c_add[1:] = 1.0 / ((1 - alpha) * K)
    a_ub_rows = np.zeros((K, n + n_extra))
    a_ub_rows[:, :n] = losses
    a_ub_rows[:, n] = -1.0
    for k in range(K):
        a_ub_rows[k, n + 1 + k] = -1.0
    b_ub = np.zeros(K)
    return c_add, a_ub_rows, b_ub, 0


def fallback_two_stage_stochastic_plan(posterior_samples, block_cost, price, *, alpha=0.9, lam=1.0):
    """H4 `two_stage_stochastic_plan(posterior, block_cost, price, *, k_scenarios, alpha, rng)
    -> StochasticPlan`: maximize E_k[v_k(x)] - lambda*CVaR_alpha(-v_k(x)) as one MILP."""
    g = posterior_samples
    K, n_blocks = g.shape
    c_kb = price * g - block_cost[None, :]
    losses = -c_kb
    c_add, a_ub_rows, b_ub, _ = fallback_cvar_epigraph(losses, alpha)
    n_extra = c_add.size
    c = np.concatenate([c_kb.mean(axis=0), -lam * c_add])
    A = np.zeros((K, n_blocks + n_extra))
    A[:, : n_blocks + n_extra] = a_ub_rows
    bounds = [(0.0, 1.0)] * n_blocks + [(-1.0e6, 1.0e6)] + [(0.0, 1.0e6)] * K
    _, sol = branch_and_bound_milp(c, A, b_ub, integer=list(range(n_blocks)), bounds=bounds, sense="max")
    extract = np.round(sol[:n_blocks]).astype(int)
    ev = float(c_kb.mean(axis=0) @ extract)
    losses_x = losses @ extract
    var_q = np.quantile(losses_x, alpha)
    tail = losses_x[losses_x >= var_q]
    cvar = float(tail.mean()) if len(tail) else float(var_q)
    return StochasticPlanFallback(extract=extract, expected_value=ev, cvar=cvar, scenarios=g)


def run_stochastic_plan(g, block_cost, price, *, alpha, lam, rng):
    """Adapter: the frozen H4 signature takes an IC-1 `Posterior`, not a raw scenario matrix, and has no
    explicit `lam` (its risk weight is presumably an internal default). We replay the fixture's committed
    scenarios through a minimal `.samples`-only stub so the *real* solver, once it lands, is exercised
    end-to-end unchanged; until then the fallback consumes the same scenarios directly."""
    if HAVE_REAL_H4:
        class _FixturePosterior:
            def samples(self, n, rng):
                return g[rng.integers(0, g.shape[0], size=n)]

        return two_stage_stochastic_plan(_FixturePosterior(), block_cost, price,
                                          k_scenarios=g.shape[0], alpha=alpha, rng=rng)
    return fallback_two_stage_stochastic_plan(g, block_cost, price, alpha=alpha, lam=lam)


plan_fn = run_stochastic_plan

## 3. The deterministic-mean baseline plan

The naive planner: extract a block if its *sample-mean* margin (over the same training scenarios) is positive, ignoring downside risk entirely.

In [ ]:
c_kb_train = price * g - cost[None, :]
x_det = (c_kb_train.mean(axis=0) > 0).astype(int)
print("deterministic-mean plan extracts blocks:", np.nonzero(x_det)[0].tolist())

## 4. The stochastic, CVaR-averse plan

In [ ]:
rng = np.random.default_rng(0)
plan = plan_fn(g, cost, price, alpha=alpha, lam=lam, rng=rng)
x_stoch = np.asarray(plan.extract)
print("stochastic CVaR-averse plan extracts blocks:", np.nonzero(x_stoch)[0].tolist())

## 5. Held-out evaluation + the risk diagnostic plot

Both plans are fixed decisions; we score them on the 400 held-out truth scenarios neither plan ever saw. This is the UQ-facing plot: the stochastic plan's value distribution has a visibly shorter downside tail.

In [ ]:
c_test = price * g_test - cost[None, :]
v_det = c_test @ x_det
v_stoch = c_test @ x_stoch


def cvar_of_losses(losses, alpha):
    q = np.quantile(losses, alpha)
    tail = losses[losses >= q]
    return float(tail.mean()) if len(tail) else float(q)


ev_det, ev_stoch = float(v_det.mean()), float(v_stoch.mean())
cvar_det = cvar_of_losses(-v_det, alpha)
cvar_stoch = cvar_of_losses(-v_stoch, alpha)

print(f"expected value  det={ev_det:.4f}  stoch={ev_stoch:.4f}")
print(f"CVaR (downside) det={cvar_det:.4f}  stoch={cvar_stoch:.4f}")

fig, ax = plt.subplots(figsize=(6.5, 3.6))
ax.hist(v_det, bins=30, alpha=0.55, color="#C0392B", label="deterministic-mean plan")
ax.hist(v_stoch, bins=30, alpha=0.55, color="#2E86AB", label="stochastic CVaR-averse plan")
ax.axvline(-cvar_det, color="#C0392B", ls="--", lw=1.4)
ax.axvline(-cvar_stoch, color="#2E86AB", ls="--", lw=1.4)
ax.set_title("held-out portfolio value -- deterministic vs. CVaR-averse plan")
ax.set_xlabel("realized value on held-out truth scenarios"); ax.legend()
plt.tight_layout(); plt.show()

## 6. Definition of Done

Reproduces H4's exec-DoD threshold on the held-out truth scenarios: the stochastic plan's expected value must be `>=` the deterministic-mean plan's, and its CVaR (downside) must be strictly better.

In [ ]:
assert ev_stoch >= ev_det, f"stochastic EV {ev_stoch:.4f} < deterministic EV {ev_det:.4f}"
assert cvar_stoch < cvar_det, f"stochastic CVaR {cvar_stoch:.4f} not better than deterministic {cvar_det:.4f}"
print(f"PASS -- EV {ev_stoch:.4f} >= {ev_det:.4f}, CVaR {cvar_stoch:.4f} < {cvar_det:.4f}")